### Bart finetune

In [ ]:
! pip install transformers datasets accelerate

In [ ]:
!pip install scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

preprocessing

In [ ]:
! cd

for later: bart-larger

In [ ]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large")

# def tokenize(example):
#     return tokenizer(example["text"], truncation=True, padding="max_length")

# dataset = dataset.map(tokenize, batched=True)
# dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

### Dataset Overview

In [1]:
import pandas as pd

# ========================
# LOAD DATASET
# ========================
file_path = "dataset/processed/emails_sms_combined.csv"  # change if needed

df = pd.read_csv(file_path)

# ========================
# BASIC INFO
# ========================
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"Total samples: {len(df)}")

# ========================
# CHANNEL COUNT
# ========================
print("\nChannel Distribution:")
channel_counts = df["channel"].value_counts()
print(channel_counts)

# ========================
# LABEL COUNT
# ========================
print("\nLabel Distribution (0 = legit, 1 = phishing):")
label_counts = df["label"].value_counts()
print(label_counts)

# ========================
# CROSS ANALYSIS
# ========================
print("\nChannel vs Label Breakdown:")
cross_tab = pd.crosstab(df["channel"], df["label"])
print(cross_tab)

# ========================
# PERCENTAGES (OPTIONAL)
# ========================
print("\nPercentage Distribution:")
percentage = pd.crosstab(df["channel"], df["label"], normalize="index") * 100
print(percentage.round(2))

# ========================
# EXTRA INSIGHTS
# ========================
print("\nPhishing Ratio:")
phishing_ratio = (df["label"] == 1).mean() * 100
print(f"{phishing_ratio:.2f}% phishing")

print("\nLegitimate Ratio:")
legit_ratio = (df["label"] == 0).mean() * 100
print(f"{legit_ratio:.2f}% legitimate")

print("=" * 50)

DATASET OVERVIEW
Total samples: 22199

Channel Distribution:
channel
email    15148
sms       7051
Name: count, dtype: int64

Label Distribution (0 = legit, 1 = phishing):
label
0    12122
1    10077
Name: count, dtype: int64

Channel vs Label Breakdown:
label       0     1
channel            
email    7153  7995
sms      4969  2082

Percentage Distribution:
label        0      1
channel              
email    47.22  52.78
sms      70.47  29.53

Phishing Ratio:
45.39% phishing

Legitimate Ratio:
54.61% legitimate


## DistilBERT multi-feature phishing detection

In [2]:
import re

def extract_features(text):
    text_lower = text.lower()

    # ========================
    # REQUEST TYPE (ROBUST)
    # ========================
    if re.search(
        r"(password|passcode|otp|one[- ]?time password|login|sign[- ]?in|verify( your)? account|credentials|authentication|security code|2fa|pin)",
        text_lower
    ):
        request_type = 1  # credentials

    elif re.search(
        r"(pay|payment|transfer|upi|bank transfer|refund|invoice|billing|transaction|debit|credit|wallet|settlement|amount due)",
        text_lower
    ):
        request_type = 2  # payment

    elif re.search(
        r"(details|information|update|confirm|submit|provide|fill|complete|registration|profile|kyc|documents|id proof)",
        text_lower
    ):
        request_type = 3  # personal info

    else:
        request_type = 0

    # ========================
    # MANIPULATION (EXPANDED)
    # ========================
    manipulation = int(bool(re.search(
        r"(urgent|immediately|now|asap|right away|action required|important notice|final warning|last chance|limited time|deadline|within \d+ (hours|days)|"
        r"suspend|suspension|account (will be )?blocked|deactivated|terminated|avoid suspension|avoid penalty|failure to act)",
        text_lower
    )))

    # ========================
    # IMPERSONATION (GENERALIZED)
    # ========================
    impersonation = int(bool(re.search(
        r"(we are|this is|on behalf of|representing|official|team|department|support|customer service|helpdesk|authority|administration|"
        r"bank|university|institute|government|service provider)",
        text_lower
    )))

    # ========================
    # INTENT (LOGIC-BASED)
    # ========================
    intent = int(
        manipulation or
        request_type != 0 or
        impersonation
    )

    return {
        "intent": intent,
        "manipulation": manipulation,
        "request_type": request_type,
        "impersonation": impersonation
    }

In [3]:
import torch
import torch.nn as nn
from transformers import DistilBertPreTrainedModel, DistilBertModel

class DistilBertMultiTaskClassifier(DistilBertPreTrainedModel):
    """Memory-optimized multi-head classifier"""

    def __init__(self, config):
        super().__init__(config)
        
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.dropout)
        
        # Multi-head classifiers
        self.intent_classifier = nn.Linear(config.hidden_size, 3)
        self.manipulation_classifier = nn.Linear(config.hidden_size, 2)
        self.request_type_classifier = nn.Linear(config.hidden_size, 4)
        self.impersonation_classifier = nn.Linear(config.hidden_size, 2)
        self.context_classifier = nn.Linear(config.hidden_size, 2)  # NEW

        # 🔥 IMPORTANT: reduce memory
        self.config.use_cache = False

        # 🔥 OPTIONAL (huge memory saver)
        self.gradient_checkpointing_enable()

    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        **kwargs
    ):
        # Handle labels from Trainer
        if labels is None and 'intent' in kwargs:
            labels = {
                'intent': kwargs.pop('intent'),
                'manipulation': kwargs.pop('manipulation'),
                'request_type': kwargs.pop('request_type'),
                'impersonation': kwargs.pop('impersonation'),
                'context': kwargs.pop('context'),
            }

        # 🔥 IMPORTANT: return_dict=False reduces overhead
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=False
        )

        # outputs[0] = last_hidden_state
        pooled_output = outputs[0][:, 0]   # CLS token (slightly faster)
        pooled_output = self.dropout(pooled_output)

        # Heads
        intent_logits = self.intent_classifier(pooled_output)
        manipulation_logits = self.manipulation_classifier(pooled_output)
        request_type_logits = self.request_type_classifier(pooled_output)
        impersonation_logits = self.impersonation_classifier(pooled_output)
        context_logits = self.context_classifier(pooled_output)  # NEW

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()

            # 🔥 Weighted loss (better learning, same memory)
            loss = (
                1.2 * loss_fn(intent_logits, labels["intent"]) +
                1.0 * loss_fn(manipulation_logits, labels["manipulation"]) +
                2.0 * loss_fn(request_type_logits, labels["request_type"]) +
                1.5 * loss_fn(impersonation_logits, labels["impersonation"]) +
                1.3 * loss_fn(context_logits, labels["context"])  # NEW
            ) / 6.0

        return {
            "loss": loss,
            "intent": intent_logits,
            "manipulation": manipulation_logits,
            "request_type": request_type_logits,
            "impersonation": impersonation_logits,
            "context": context_logits,
        }

/mnt/DISK/Studies/SEM 4/Project Course 299/Phishing-detector/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd
import torch
import gc
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

print("loading")

# ========================
# SAFE CSV LOAD (VERY IMPORTANT)
df = pd.read_csv(
    "dataset/processed/emails_sms_combined.csv",
    engine="python",
    on_bad_lines="skip"
)

df = df[["channel", "text", "label"]]

# ========================
# CONTEXT WEAK SUPERVISION
def weak_context_label(text):
    text = text.lower()
    keywords = [
        "dear all", "this is to inform", "maintenance",
        "campus", "facility", "students", "staff",
        "inspection", "schedule", "administration",
        "department", "lab", "office"
    ]
    return 1 if any(k in text for k in keywords) else 0

# NEW context labels for institutional detection
df["context"] = df["text"].apply(weak_context_label)

# 🔥 FIX LABEL ISSUES
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

print(df.head())
print("Dataset size:", len(df))

# ========================
# CONVERT TO DATASET
dataset = Dataset.from_pandas(df)

# ========================

def preprocess(example):
    features = extract_features(example["text"])

    # ONLY fix intent — do NOT overwrite everything
    if example["label"] == 1:
        features["intent"] = 1
    else:
        features["intent"] = 0

    return {
        "text": example["text"],
        "channel": example["channel"],
        "context": example["context"],
        **features
    }

dataset = dataset.map(preprocess)

# ========================
# TOKENIZER

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

dataset.reset_format()
cols_to_keep = ["text", "channel", "intent", "manipulation", "request_type", "impersonation", "context"]
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in cols_to_keep])

def tokenize(example):
    text = example["text"]
    return tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

# ❌ REMOVE keep_in_memory=True (VERY IMPORTANT)
dataset = dataset.map(tokenize, batched=False)

# ========================
# FINAL FORMAT
dataset = dataset.remove_columns([
    col for col in dataset.column_names 
    if col not in ["input_ids", "attention_mask", 
                   "intent", "manipulation", 
                   "request_type", "impersonation", "context"]
])

# ========================
# SPLIT
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

# ========================
# CLEAN GPU BEFORE MODEL LOAD
gc.collect()
torch.cuda.empty_cache()

# ========================
# LOAD MODEL (DO NOT MOVE TO DEVICE)
config = AutoConfig.from_pretrained("distilbert-base-uncased")
model = DistilBertMultiTaskClassifier(config)

# ❌ REMOVE THIS:
# model.to(device)

# ========================
# METRICS (SAFE VERSION)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    logits = predictions[0]

    intent_preds = logits["intent"].argmax(axis=1)
    manipulation_preds = logits["manipulation"].argmax(axis=1)
    request_type_preds = logits["request_type"].argmax(axis=1)
    impersonation_preds = logits["impersonation"].argmax(axis=1)
    context_preds = logits["context"].argmax(axis=1)

    return {
        "intent_accuracy": accuracy_score(labels["intent"], intent_preds),
        "intent_f1": f1_score(labels["intent"], intent_preds, average="weighted", zero_division=0),
        "manipulation_accuracy": accuracy_score(labels["manipulation"], manipulation_preds),
        "request_type_accuracy": accuracy_score(labels["request_type"], request_type_preds),
        "impersonation_accuracy": accuracy_score(labels["impersonation"], impersonation_preds),
        "context_accuracy": accuracy_score(labels["context"], context_preds),
    }


loading
  channel                                               text  label  context
0   email  We detected malware activity on your workstati...      1        0
1     sms  Critical virus detected on device. Pay [Amount...      1        0
2   email  Our security team identified your IP participa...      1        0
3     sms  Ransomware installed. Send [Amount] in BTC to ...      1        0
4   email  Your website has been injected with remote acc...      1        0
Dataset size: 22193


Map: 100%|██████████| 22193/22193 [00:22<00:00, 984.89 examples/s] 


In [5]:
import torch
from transformers import DataCollatorWithPadding, Trainer

# ========================
# CUSTOM DATA COLLATOR
class MultiTaskDataCollator:
    def __init__(self, tokenizer):
        self.data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    
    def __call__(self, batch):
        labels = {
            "intent": [],
            "manipulation": [],
            "request_type": [],
            "impersonation": [],
            "context": []
        }

        input_features = []

        for item in batch:
            # ✅ Do NOT mutate original dataset item
            item_copy = item.copy()

            labels["intent"].append(item_copy.pop("intent"))
            labels["manipulation"].append(item_copy.pop("manipulation"))
            labels["request_type"].append(item_copy.pop("request_type"))
            labels["impersonation"].append(item_copy.pop("impersonation"))
            labels["context"].append(item_copy.pop("context"))

            input_features.append(item_copy)

        batch_data = self.data_collator(input_features)

        # Convert labels to tensors
        for key in labels:
            batch_data[key] = torch.tensor(labels[key], dtype=torch.long)

        return batch_data


# ========================
# CUSTOM TRAINER
class MultiTaskTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Extract labels
        intent_labels = inputs.pop("intent")
        manipulation_labels = inputs.pop("manipulation")
        request_type_labels = inputs.pop("request_type")
        impersonation_labels = inputs.pop("impersonation")
        context_labels = inputs.pop("context")

        # Forward pass
        outputs = model(**inputs)

        # Loss
        loss_fn = torch.nn.CrossEntropyLoss()
        loss = (
            loss_fn(outputs["intent"], intent_labels) +
            loss_fn(outputs["manipulation"], manipulation_labels) +
            loss_fn(outputs["request_type"], request_type_labels) +
            loss_fn(outputs["impersonation"], impersonation_labels) +
            loss_fn(outputs["context"], context_labels)
        ) / 5.0

        return (loss, outputs) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):

        # Extract labels
        labels = {
            "intent": inputs.pop("intent"),
            "manipulation": inputs.pop("manipulation"),
            "request_type": inputs.pop("request_type"),
            "impersonation": inputs.pop("impersonation"),
            "context": inputs.pop("context")
        }

        # Disable gradients
        with torch.no_grad():
            outputs = model(**inputs)

        loss = outputs.get("loss")

        # 🔥 Move loss to CPU (important)
        if loss is not None:
            loss = loss.detach().cpu()

        # 🔥 Respect Trainer setting
        if prediction_loss_only:
            return loss, None, None

        # 🔥 Move logits to CPU immediately (prevents OOM)
        logits = {
            k: v.detach().cpu()
            for k, v in outputs.items() if k != "loss"
        }

        # 🔥 Move labels to CPU
        labels = {
            k: v.detach().cpu()
            for k, v in labels.items()
        }

        return loss, logits, labels


# ========================
# INIT COLLATOR
data_collator = MultiTaskDataCollator(tokenizer)


In [6]:
# Debug: Check dataset columns
print("Train dataset columns:", train_dataset.column_names)
print("Train dataset sample:")
print(train_dataset[0])

Train dataset columns: ['context', 'intent', 'manipulation', 'request_type', 'impersonation', 'input_ids', 'attention_mask']
Train dataset sample:
{'context': 0, 'intent': 1, 'manipulation': 0, 'request_type': 0, 'impersonation': 0, 'input_ids': [101, 18777, 1024, 21423, 20600, 2326, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [7]:
training_args = TrainingArguments(
    output_dir="./distilbert-phishing-model",

    # 🔥 CORE
    per_device_train_batch_size=4,   # safer for 6GB GPU
    per_device_eval_batch_size=2,    # VERY IMPORTANT (prevents OOM)

    num_train_epochs=2,

    # 🔥 GPU SETTINGS
    fp16=False,  # keep False for stability on your setup

    # 🔥 MEMORY CONTROL
    eval_accumulation_steps=1,       # prevents GPU buildup
    dataloader_pin_memory=True,

    # 🔥 TRAINER BEHAVIOR
    remove_unused_columns=False,    # REQUIRED for your custom labels

    # 🔥 SAVE / LOGGING
    save_strategy="no",
    logging_strategy="steps",
    logging_steps=50,

    # 🔥 EVAL
    # evaluation_strategy="epoch",
    do_eval=True,
    # ❌ REMOVE THIS:
    # use_cpu=True
)

In [8]:
# ========================
# INSTANTIATE TRAINER
# ========================
trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()
print("Training complete!")
# ========================
# SAFE EVALUATION
# ========================
import torch
import gc

try:
    print("Preparing for evaluation...")

    # 🔥 Clean GPU completely
    gc.collect()
    torch.cuda.empty_cache()

    # 🔥 Disable cache (important for transformers)
    model.config.use_cache = False

    print("Starting evaluation...")

    results = trainer.evaluate(
        eval_dataset=val_dataset,
        metric_key_prefix="eval"
    )

    print("\nEvaluation results:")
    for k, v in results.items():
        print(f"{k}: {v}")

except RuntimeError as e:
    print("\n⚠️ CUDA issue detected:", str(e))

    print("\n🔄 Retrying with ultra-safe settings...")

    # 🔥 fallback: smallest possible batch
    trainer.args.per_device_eval_batch_size = 1

    gc.collect()
    torch.cuda.empty_cache()

    results = trainer.evaluate(
        eval_dataset=val_dataset,
        metric_key_prefix="eval"
    )

    print("\nRecovered Evaluation results:")
    for k, v in results.items():
        print(f"{k}: {v}")

except Exception as e:
    print("\n⚠️ Evaluation completed but callback failed:", str(e))

    # fallback logs
    if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
        last_log = trainer.state.log_history[-1] if trainer.state.log_history else {}
        print("Last log:", last_log)

Starting training...


Step,Training Loss
50,0.830194
100,0.717005
150,0.699848
200,0.637295
250,0.671025
300,0.579364
350,0.592527
400,0.559230
450,0.524526
500,0.559839


Training complete!
Preparing for evaluation...
Starting evaluation...



⚠️ Evaluation completed but callback failed: 0
Last log: {'train_runtime': 710.2008, 'train_samples_per_second': 56.246, 'train_steps_per_second': 14.064, 'total_flos': 1305026693137920.0, 'train_loss': 0.32465689775224776, 'epoch': 2.0, 'step': 9988}


In [9]:
model.save_pretrained("./distilbert-phishing-model")
tokenizer.save_pretrained("./distilbert-phishing-model")

# 🔥 Save class info manually (important for custom model)
model.config.architectures = ["DistilBertMultiTaskClassifier"]
model.config.save_pretrained("./distilbert-phishing-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.05it/s]


In [10]:
import torch
import gc
from safetensors.torch import load_file
from transformers import AutoConfig

# ========================
# CLEAN MEMORY FIRST
# ========================
gc.collect()
torch.cuda.empty_cache()

# ========================
# LOAD MODEL (CPU FIRST)
# ========================
config = AutoConfig.from_pretrained("distilbert-base-uncased")
model = DistilBertMultiTaskClassifier(config)

# Load weights safely on CPU
state_dict = load_file("./distilbert-phishing-model/model.safetensors")
model.load_state_dict(state_dict)

model.eval()

# ========================
# MOVE TO GPU (SAFE)
# ========================
if torch.cuda.is_available():
    try:
        print("Moving model to GPU...")
        model = model.to("cuda")
    except RuntimeError as e:
        print("⚠️ GPU memory issue, staying on CPU:", e)
        model = model.to("cpu")

Moving model to GPU...


# load back

In [11]:
import torch
import gc
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file

model_path = "./distilbert-phishing-model"


# ========================
# CLEAN MEMORY
# ========================
gc.collect()
torch.cuda.empty_cache()

# ========================
# LOAD TOKENIZER + CONFIG
# ========================
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)

# ========================
# INIT MODEL (CPU FIRST)
# ========================
model = DistilBertMultiTaskClassifier(config)

# Load weights safely
state_dict = load_file(f"{model_path}/model.safetensors")
model.load_state_dict(state_dict)

model.eval()

# ========================
# OPTIONAL: MOVE TO GPU
# ========================
if torch.cuda.is_available():
    try:
        print("Using GPU")
        model = model.to("cuda")
    except RuntimeError as e:
        print("⚠️ GPU not available, using CPU:", e)

Using GPU


In [17]:
import torch
import torch.nn.functional as F

# ========================
# LABEL MAPPINGS
# ========================
INTENT_LABELS = {0: "normal", 1: "phishing", 2: "suspicious"}
REQUEST_TYPE_LABELS = {0: "none", 1: "credentials", 2: "payment", 3: "personal_info"}
BINARY_LABELS = {0: "no", 1: "yes"}
CONTEXT_LABELS = {0: "user_targeted", 1: "institutional"}

FINAL_LABEL_MAP = {
    0: "legitimate",
    1: "suspicious",
    2: "phishing"
}


# ========================
# FINAL LABEL LOGIC
# ========================
def get_final_label(score, intent_idx, intent_confidence):

    # 🔴 strong phishing ONLY if intent = phishing
    if intent_idx == 1 and intent_confidence > 0.85:
        return 2

    # 🔴 strong phishing by score
    if score >= 6:
        return 2

    # 🟡 suspicious
    if score >= 3:
        return 1

    # 🟢 legitimate
    return 0


# ========================
# ANALYSIS FUNCTION
# ========================
def analyze_message(text, channel="email"):
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # ========================
    # LOGITS
    intent_logits = outputs["intent"]
    manipulation_logits = outputs["manipulation"]
    request_type_logits = outputs["request_type"]
    impersonation_logits = outputs["impersonation"]
    context_logits = outputs["context"]

    # ========================
    # PROBABILITIES
    intent_probs = F.softmax(intent_logits, dim=1)
    manipulation_probs = F.softmax(manipulation_logits, dim=1)
    request_type_probs = F.softmax(request_type_logits, dim=1)
    impersonation_probs = F.softmax(impersonation_logits, dim=1)
    context_probs = F.softmax(context_logits, dim=1)

    # ========================
    # PREDICTIONS
    intent_idx = intent_probs.argmax(dim=1).item()
    manipulation_idx = manipulation_probs.argmax(dim=1).item()
    request_type_idx = request_type_probs.argmax(dim=1).item()
    impersonation_idx = impersonation_probs.argmax(dim=1).item()
    context_idx = context_probs.argmax(dim=1).item()

    # ========================
    # CONFIDENCE
    intent_confidence = intent_probs[0][intent_idx].item()
    manipulation_confidence = manipulation_probs[0][manipulation_idx].item()
    request_type_confidence = request_type_probs[0][request_type_idx].item()
    impersonation_confidence = impersonation_probs[0][impersonation_idx].item()
    context_confidence = context_probs[0][context_idx].item()

    # ========================
    # SCORING
    score = (
    3 * intent_probs[0][1] +              
    2 * manipulation_probs[0][1] +
    3 * request_type_probs[0][1] +
    2 * request_type_probs[0][2] +
    1 * request_type_probs[0][3] +
    2 * impersonation_probs[0][1]
).item()

    if request_type_probs[0][1] < 0.3 and manipulation_probs[0][1] < 0.4:
        score -= 2

    if request_type_idx == 3 and manipulation_idx == 0:
        score -= 1.5

    if context_idx == 1 and context_confidence > 0.7:
        score -= 4

    # Prevent negative score
    score = max(score, 0)

    # ========================
    # FINAL LABEL
    final_label = get_final_label(score, intent_idx, intent_confidence)

    # ========================
    # EXPLANATIONS (OPTIONAL 🔥)
    reasons = []

    if manipulation_idx == 1 and context_idx == 0:
        reasons.append("urgent/manipulative tone")

    if request_type_idx == 1:
        reasons.append("requests credentials")

    if request_type_idx == 2:
        reasons.append("requests payment")

    if impersonation_idx == 1 and context_idx == 0:
        reasons.append("possible impersonation")

    # ========================
    # RETURN
    return {
        "text": text[:100] + "..." if len(text) > 100 else text,
        "channel": channel,

        "intent": {
            "prediction": INTENT_LABELS[intent_idx],
            "confidence": intent_confidence
        },
        "manipulation": {
            "prediction": BINARY_LABELS[manipulation_idx],
            "confidence": manipulation_confidence
        },
        "request_type": {
            "prediction": REQUEST_TYPE_LABELS[request_type_idx],
            "confidence": request_type_confidence
        },
        "impersonation": {
            "prediction": BINARY_LABELS[impersonation_idx],
            "confidence": impersonation_confidence
        },
        "context": {
            "prediction": CONTEXT_LABELS[context_idx],
            "confidence": context_confidence
        },

        "score": round(score, 2),
        "max_score": 10,
        "final_label": FINAL_LABEL_MAP[final_label],

        "reasons": reasons
    }


# ========================
# DEMO
print("=" * 60)
print("PHISHING DETECTION ANALYSIS")
print("=" * 60)

test_messages = [
    ("""Hello How are you? This is a friendly reminder about your upcoming appointment. Please let us know if you have any questions. Thank you!

""", "sms")
]

for msg, ch in test_messages:
    result = analyze_message(msg, channel=ch)

    print(f"\nMessage: {result['text']}")
    print(f"Channel: {result['channel']}")

    print(f"Intent: {result['intent']['prediction']} (conf: {result['intent']['confidence']:.2f})")
    print(f"Manipulation: {result['manipulation']['prediction']} (conf: {result['manipulation']['confidence']:.2f})")
    print(f"Request Type: {result['request_type']['prediction']} (conf: {result['request_type']['confidence']:.2f})")
    print(f"Impersonation: {result['impersonation']['prediction']} (conf: {result['impersonation']['confidence']:.2f})")
    print(f"Context: {result['context']['prediction']} (conf: {result['context']['confidence']:.2f})")

    print(f"Risk Score: {result['score']} / {result['max_score']}")
    print(f"Final Label: {result['final_label'].upper()}")

    if result["reasons"]:
        print(f"Reasons: {', '.join(result['reasons'])}")

    print("-" * 60)


PHISHING DETECTION ANALYSIS

Message: Hello How are you? This is a friendly reminder about your upcoming appointment. Please let us know i...
Channel: sms
Intent: normal (conf: 0.90)
Manipulation: yes (conf: 1.00)
Request Type: none (conf: 1.00)
Impersonation: no (conf: 0.93)
Context: user_targeted (conf: 1.00)
Risk Score: 2.46 / 10
Final Label: LEGITIMATE
Reasons: urgent/manipulative tone
------------------------------------------------------------
